In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
from pathlib import Path
import sys
import html as html_lib
from IPython.display import HTML, display as ipy_display
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns


root = Path.cwd().resolve()
root = root.parent
sys.path.insert(0, str(root))

import torch
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM
from sae_trainer.core.model_utils import SparseAutoencoder, ReluSparseAutoencoder, TopKSparseAutoencoder
from sae_trainer.sae_semantic_eval.feature_tracer_utils import FeatureTracer, TraceConfig
from sae_trainer.sae_semantic_eval.feature_tracer_eval_utils import (
    search_features_by_concept, 
    build_query_feature_matrix,
    build_query_feature_matrix, 
    compare_query_differences,
    render_feature_card,
    build_query_feature_matrix, 
    compare_query_differences, 
    explain_difference_similarity,
    top_n_feats_by_hits_count, 
    top_n_feats_by_act_mass, 
    prompt_x_feature_heatmap, 
    feature_token_position_profile, 
    top_m_feature_coactivation, 
    render_feature_card, 
    plot_feature_umap, 
    plot_inter_feature_similarity, 
    plot_feature_similarity_violins,
    build_token_feature_spectrum
)
import umap
from sklearn.cluster import HDBSCAN  # or: from hdbscan import HDBSCAN
from datasets import load_dataset



/Users/alyssagardiner/src/sae_trainer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
#device = "cuda" if torch.cuda.is_available() else "cpu"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")
layer_idx = 12
dataset_name = 'openwebtext'
model_name_simple = "qwen" # Either gpt2 or qwen
sae_type = 'topk'
k = 40

if model_name_simple == "gpt2":
    model_name = "gpt2"
elif model_name_simple == "qwen":
    model_name = "Qwen/Qwen2-0.5B-Instruct"
else:
    raise ValueError(f"Invalid model name: {model_name_simple}. Expected either 'gpt2' or 'qwen'.")
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()

# Load SAE checkpoint — pick whichever layer you want to analyse (3, 6, 9, or 11)
ckpt = torch.load(root / f"model_weights_files/{sae_type}_sae_{model_name_simple}_{dataset_name}_layer{layer_idx}.pt", map_location=device)
if sae_type == 'relu':
    sae = ReluSparseAutoencoder(d_in=ckpt["d_in"], d_latent=ckpt["d_latent"], normalize_decoder=True).to(device)
elif sae_type == 'topk':
    sae = TopKSparseAutoencoder(d_in=ckpt["d_in"], d_latent=ckpt["d_latent"], k=k, normalize_decoder=True).to(device)
else:
    raise ValueError(f"Invalid sae type: {sae_type}. Expected either 'relu' or 'qwen'.")
sae.load_state_dict(ckpt["model_state"])
sae.eval()

device: mps


TopKSparseAutoencoder(
  (encoder): Linear(in_features=896, out_features=7168, bias=True)
  (decoder): Linear(in_features=7168, out_features=896, bias=False)
)

In [17]:
# 4) Tracer config — layer_idx MUST match the checkpoint loaded above
cfg = TraceConfig(
    layer_idx=layer_idx,
    topk_per_token=8,
    min_activation=0.01,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    token_mode='generated_only',
    min_prompts=5
)
primary_tracer = FeatureTracer(llm, tokenizer, sae, device, cfg)

In [18]:
n_train_docs = 4000

ds = load_dataset("openwebtext", split="train", streaming=True)
ds = ds.skip(n_train_docs)

summaries = primary_tracer.trace_prompts_from_iterable_dataset(ds, max_prompts_to_trace=750)

/Users/alyssagardiner/src/sae_trainer/.venv/lib/python3.12/site-packages/transformers/pytorch_utils.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [19]:
scores = primary_tracer.feature_specificity_scores()

In [10]:
raw_queries = [
    'Sadness',
    'Anger',
    'Happiness',
    'Fear',
    'Surprise',
    'Disgust',
    'Pride',
    'Gratefulness',
    'Longing for',
    'Despise',
    'Despair',
    'Shame',
    'Guilt',
    'Disappointment',
    'Pleasure',
    'Nervousness',
    'Winning',
    'Losing',
    'Succeeding',
    'Failing',
    'Wanting',
    'Craving',
    'Waiting',
    'Inhibiting',
    'Social longing',
    'Speaking',
    'Laughing',
    'Love',
    'Social exclusion',
    'Togetherness',
    'Loneliness',
    'Ostracism',
    'Closeness (in social relations)',
    'Hostility',
    'Sympathy',
    'Anxiety',
    'Panic',
    'Stress',
    'Depression',
    'Mania',
    'Eating',
    'Drinking',
    'Hunger',
    'Satiation',
    'Thirst',
    'Sweating',
    'Urinating',
    'Defecation',
    'Tiredness',
    'Vitality',
    'Exhaustion',
    'Breathing',
    'Heartbeat',
    'Drunkenness',
    'Sexual arousal',
    'Orgasm',
    'Shivering',
    'Hangover',
    'Heartburn',
    'Suffocation',
    'Relaxation',
    'Sleeping',
    'Thinking',
    'Remembering',
    'Attending',
    'Forgetting',
    'Reading',
    'Imagining',
    'Daydreaming',
    'Inferring',
    'Reasoning',
    'Estimating',
    'Being conscious',
    'Memorizing',
    'Recognition',
    'Recollection',
    'Having flu',
    'Having fever',
    'Having stomach flu',
    'Having cold',
    'Having headache',
    'Feeling nauseous',
    'Having toothache',
    'Coughing',
    'Sneezing',
    'Aching',
    'Being dazzled',
    'Seeing',
    'Hearing',
    'Feeling touch',
    'Smelling',
    'Tasting',
    'Itching',
    'Numbness',
    'Feeling pain',
    'Hotness',
    'Coldness',
    'Dizziness',
    'Movement',
    'Acceleration'
]

queries = [f"Describe the following emotion: {raw_query}" for raw_query in raw_queries]

In [11]:
feature_matrix, sim_matrix = build_query_feature_matrix(queries, primary_tracer)

# feature_matrix: (n_queries, n_features) — each row is a query's SAE fingerprint
# sim_matrix:     (n_queries, n_queries)          — pairwise similarity between queries

pd.DataFrame(sim_matrix, index=queries, columns=queries).round(3)

sns.heatmap(sim_matrix)

NameError: name 'primary_tracer' is not defined

In [ ]:

# token_mode="generated_only" means the spectrum reflects only the response,
# not the prompt tokens
config = TraceConfig(layer_idx=layer_idx, token_mode="generated_only", max_new_tokens=50)
query_tracer = FeatureTracer(llm, tokenizer, sae, device, config)
query_tracer.trace_prompts(queries, ids=list(range(len(queries))))

In [ ]:
n_features = sae.d_latent

first_token_matrix = np.stack([
    build_token_feature_spectrum(i, query_tracer, n_features).iloc[0].values
    for i in range(len(queries))
]).astype(np.float32)  # (n_queries, n_features)

In [ ]:
results = []
for n_dim in [2, 3, 5, 8, 10, 15, 20]:
    for min_cluster_size in [3, 5, 8]:
        reducer = umap.UMAP(n_components=n_dim, metric="cosine", random_state=42)
        embedding = reducer.fit_transform(first_token_matrix)

        clusterer = HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean")
        labels = clusterer.fit_predict(embedding)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)

        # DBCV-style validity index — HDBSCAN-native, works with non-convex clusters
        validity = validity_index(embedding.astype("float64"), labels, metric="euclidean")

        results.append({
            "n_dim": n_dim,
            "min_cluster_size": min_cluster_size,
            "n_clusters": n_clusters,
            "n_noise": n_noise,
            "validity": validity,
        })

for r in results:
    print(r)

In [ ]:
reducer_cluster = umap.UMAP(n_components=8, metric="cosine", random_state=42)
embedding_cluster = reducer_cluster.fit_transform(first_token_matrix)

clusterer = HDBSCAN(min_cluster_size=3, metric="euclidean")
labels = clusterer.fit_predict(embedding_cluster)

# For visualization only — separate 2D projection
reducer_viz = umap.UMAP(n_components=2, metric="cosine", random_state=42)
coords = reducer_viz.fit_transform(first_token_matrix)

plt.figure(figsize=(12, 9))
for cid in sorted(set(labels)):
    mask = np.array(labels) == cid
    label_name = "Noise" if cid == -1 else f"Cluster {cid}"
    color = "lightgray" if cid == -1 else None
    plt.scatter(coords[mask, 0], coords[mask, 1], label=label_name, s=80, c=color)
for i, q in enumerate(queries):
    plt.annotate(q, coords[i], fontsize=9, ha="center", va="bottom")
plt.legend()
plt.tight_layout()
plt.show()